# Uso rápido RobusPredictor 
Para mayor apoyo consultar RobusPredictor.md

Versión librería: 1.3.0
Autores: Sebastián Valdivia · Paula Frías  

---

## 1. Importación del modelo

Se requiere que este jupiter y la carpeta robuspredictor se encuentren a la misma altura de directorio.

In [ ]:
from robuspredictor import RobusPredictor
# se necesita pandas para el proximo bloque
import pandas as pd

## 2. Cargar datos a utilizar

In [ ]:
ENTRENAMIENTO = pd.read_csv("/ingrese_ruta.csv")
VALIDACION = pd.read_csv("/ingrese_ruta.csv")

features = [
    "var1", "var2", 
]
target = "INTENSIDAD_4H"

X_train = ENTRENAMIENTO[features]
y_train = ENTRENAMIENTO[target]

X_valid = VALIDACION[features]
y_valid = VALIDACION[target]

ARRIENDO_REAL= VALIDACION["ARRIENDO"]

## 3. Creación del modelo 
Podemos especificar cada parametro a ocupar de robus

In [ ]:
modelo = RobusPredictor(
    n_min=20,
    n_max=500,
    n_dom=2,
    mean_min=0.0,
    mean_max=3.0,
    std_min=0.0,
    std_max=0.5,
    use_default_value=True,
    default_value=0,
    verbose=False
)

Se puede ocupar sin especificar cada parametro pero seguir el orden en que se llaman

In [ ]:
modelo = RobusPredictor(
    20,
    500,
    2,
    0.0,
    3.0,
    0.0,
    0.5,
    True,
    0,
    False
)

Y para que quede más limpio, no ocupar las variables defaults que son `use_default_value`, `default_value` y `verbose`.

In [ ]:
modelo = RobusPredictor(
    20,
    500,
    2,
    0.0,
    3.0,
    0.0,
    0.5
)

## 4. Entrenamiento y predicción

In [ ]:
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_valid)

## 5. Exportar checkpoints y scoring 

In [ ]:
checkpoint = modelo.export_checkpoint(
    X_valid=X_valid,
    y_valid=y_valid,
    file_name="checkpoint_validacion",
    file_format="xlsx"
)


scoring_df = modelo.export_prediction_checkpoint(
    X_valid=X_valid,
    y_valid=y_valid,
    dato_real=ARRIENDO_REAL,
    file_name="scoring_validacion",
    file_format="xlsx"
)

## 6. Dataframe: Resumen de cubos utilizados en predicción

In [ ]:
cubes_df = modelo.export_dataframe_cubes()

display(cubes_df.head())

## 7. Dataframe: Grilla aprendida en el entrenamiento

In [ ]:
cube_grid = modelo.export_cubes_grid()
display(cube_grid.head())

## 8. Calcular el Top 5%

In [ ]:
top_5 = modelo.best_percentage(
    y_target=ARRIENDO_REAL,
    top_pct=0.05
)

print(f"Precisión Top 5%: {top_5:.2%}")

## 9. Funcion predict_cubes
Nos entregará dentro de un nuevo dataframe que en este caso definiremos como resultado, el ID a cual pertenece cada registro. 

In [ ]:
resultado = X_valid.copy()

resultado["cube_id"] = modelo.predict_cubes(X_valid)

resultado.head()

## 10. Tips 
En caso que querramos agregar el valor de prediccion a nuestro dataframe, basta con definir una nueva columna de pandas, en este caso la llamaremos "pred".

In [ ]:
resultado["pred"] = modelo.predict(X_valid)

resultado.head()

Como `.predict()` retorna un `pd.Series`, es necesario agregarlo como si fuera una nueva columna, de esta forma queda con un encabezado que ayuda a la legibilidad de los datos.

## 11. Ejemplo completo de uso

In [ ]:
from robuspredictor import RobusPredictor 

ENTRENAMIENTO = pd.read_csv("/ingrese_ruta.csv")
VALIDACION = pd.read_csv("/ingrese_ruta.csv")

features = [
    "var1", "var2", 
]
target = "INTENSIDAD_4H"

X_train = ENTRENAMIENTO[features]
y_train = ENTRENAMIENTO[target]

X_valid = VALIDACION[features]
y_valid = VALIDACION[target]

ARRIENDO_REAL= VALIDACION["ARRIENDO"]

modelo = RobusPredictor( 
    n_min=20, 
    n_max=500, 
    mean_min=0.0, 
    mean_max=3.0, 
    std_min=0.0, 
    std_max=0.5, 
    n_dom=2, 
    use_default_value=True, 
    default_value=0, 
    verbose=False 
) 

modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_valid)

cube_ids = modelo.predict_cubes(X_valid)
resultado = X_valid.copy()

resultado["pred"] = y_pred 
resultado["cube_id"] = cube_ids 

cubes_df = modelo.export_dataframe_cubes()
cubes_df.to_csv("cubes_df.csv", index=False)

cube_grid = modelo.export_cubes_grid()
cube_grid.to_csv("grilla.csv", index=False)

top_5 = modelo.best_percentage( 
    y_target=ARRIENDO_REAL,
    top_pct=0.05
) 
checkpoint = modelo.export_checkpoint( 
    X_valid=X_valid, 
    y_valid=y_valid, 
    file_name="checkpoint_validacion", 
    file_format="xlsx"
) 

scoring_df = modelo.export_prediction_checkpoint( 
    X_valid=X_valid, 
    y_valid=y_valid, 
    dato_real=ARRIENDO_REAL, 
    file_name="scoring_validacion", 
    file_format="xlsx" 
)

## 12. Atributos
Despues de entrenarse RobusPredictor, el modelo guarda información sobre el entrenamiento, cortes, cubos y predicciones.
Estos atributos permiten inspeccionar de forma rápida como esta funcionando el modelo sin tener que abrir directamente los archivos exportados.
Tener en consideración que de acuerdo a la documentación de "RobusPredictor.md" algunos atributos se pueden generar una vez hecho predicciones o exports.

In [ ]:
print(f"Modelo entrenado       : {modelo.is_fitted}")
print(f"Variables utilizadas   : {modelo.feature_names}")
print(f"Dominios generados     : {len(modelo.domains)}")
print(f"Cortes generados       : {len(modelo.cuts)}")
print(f"Cubos estables         : {len(modelo.stable_cubes)}")
print(f"Zonas rojas            : {len(modelo.red_zones)}")
print(f"Últimas predicciones   : {len(modelo.last_predictions)}")
print(f"Checkpoint disponible  : {list(modelo.checkpoint.keys())}")